# 02 · Loading and validating an agricultural AOI

Vibes Lite reads field boundaries from GeoJSON vectors (`data.load_aoi`), from
an inline GeoJSON payload (`data.load_geojson`), and checks how a boundary sits
over a raster before a single pixel is analysed.

In [ ]:
%matplotlib inline
# --- Vibes Lite bootstrap -----------------------------------------------------
# These notebooks drive the processing engines directly (no HTTP layer needed).
import sys
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))

import config, data, geospatial, preprocessing, indices, analysis
import opencv as vl_cv          # Vibes Lite's own engine (it wraps cv2 internally)
import segmentation, timeseries, visualization, storage

PATHS = config.ensure_directories()
NOTEBOOK_OUT = PATHS["outputs"] / "notebooks"
NOTEBOOK_OUT.mkdir(parents=True, exist_ok=True)

print("project root :", ROOT)
print("raw data     :", PATHS["raw_data"].relative_to(ROOT))
print("notebook out :", NOTEBOOK_OUT.relative_to(ROOT))

In [ ]:
DEMO_NAME = "synthetic_fields"


def demo_scene(name: str = DEMO_NAME, size: int = 900):
    """Return (raster, aoi, csv) for the labelled synthetic demo scene.

    The scene is only generated when missing. It is explicitly synthetic - its
    GeoTIFF tags carry SYNTHETIC=true - so nothing here pretends to be a real
    satellite acquisition.
    """
    raster = PATHS["raw_data"] / f"{name}.tif"
    aoi = PATHS["raw_data"] / f"{name}.geojson"
    table = PATHS["raw_data"] / f"{name}_ndvi_history.csv"
    if not raster.exists() or not aoi.exists():
        scene = data.generate_synthetic_scene(name=name, size=size, overwrite=True)
        return scene["paths"]["raster"], scene["paths"]["aoi"], scene["paths"]["table"]
    return raster, aoi, table


RASTER, AOI_FILE, HISTORY_CSV = demo_scene()
print("raster :", RASTER.name)
print("aoi    :", AOI_FILE.name)
print("history:", HISTORY_CSV.name)

## The demo AOI

Four plots in UTM zone 33N. `load_aoi` reports the CRS it found - it never silently reprojects.

In [ ]:
aoi = data.load_aoi(AOI_FILE)
print("name      :", aoi["name"])
print("features  :", aoi["feature_count"])
print("area (ha) :", round(aoi["area_ha"], 2))
print("warnings  :", aoi["warnings"] or "none")
print("crs       :", geospatial.crs_info(aoi["gdf"].crs))

In [ ]:
plots = aoi["gdf"].copy()
plots["area_ha"] = plots.geometry.area / 10_000.0
plots[["name", "crop", "management", "area_ha"]].round(2)

## Drawing the boundaries

GeoPandas renders straight to matplotlib, so the notebook and the dashboard share the same geometry.

In [ ]:
figure, axes = plt.subplots(figsize=(6.5, 6.5))
plots.plot(ax=axes, column="crop", edgecolor="#0b2318", linewidth=1.4, alpha=0.55, legend=True,
           legend_kwds={"loc": "lower right", "fontsize": 9, "title": "crop"})
axes.set_title("Demo field boundaries (UTM 33N)")
axes.set_xlabel("easting (m)")
axes.set_ylabel("northing (m)")
plt.tight_layout()
plt.show()

## Validating a boundary you supply

An inline GeoJSON parcel (EPSG:4326) is validated, its area computed in hectares, and stored back into `data/raw` for reuse. Saving never overwrites an existing upload - a numeric suffix is added instead, so re-running this notebook cannot destroy your own boundaries.

In [ ]:
parcel = {
    "type": "Feature",
    "properties": {"name": "Notebook parcel", "crop": "maize", "management": "conventional"},
    "geometry": {
        "type": "Polygon",
        "coordinates": [[[12.050, 40.885], [12.100, 40.885], [12.100, 40.925], [12.050, 40.925], [12.050, 40.885]]],
    },
}
custom = data.load_geojson(parcel)
print("features  :", custom["feature_count"])
print("area (ha) :", round(custom["area_ha"], 2))
print("crs       :", custom["crs"])
print("warnings  :", custom["warnings"] or "none")

In [ ]:
stored = data.save_aoi(custom["gdf"], "notebook_parcel")
print("stored as:", stored["name"], "| bytes:", stored["bytes"])
print("vectors in data/raw:", sorted(entry["name"] for entry in data.list_datasets() if entry["type"] == "vector"))

## Does the AOI actually cover the raster?

`raster_aoi_overlap` warns early when a boundary is in a different CRS, sits outside the scene, or only clips a sliver - the most common real-world mistake.

In [ ]:
metadata = data.get_raster_metadata(RASTER)
overlap = geospatial.raster_aoi_overlap(aoi["gdf"], metadata, geospatial.get_crs(RASTER))
print("raster crs       :", metadata["crs"]["name"])
print("overlap fraction :", overlap["overlap_fraction"])
print("warnings         :", overlap["warnings"] or "none")

**Note.** Boundaries in a different CRS are reprojected on the fly
(`geospatial.reproject_aoi`), and boundaries that miss the raster raise a
warning instead of silently returning empty statistics.